# 30 — Baselines de prevalencia y heurística

Establece referencias reproducibles antes de entrenar modelos. Este notebook **no consume TEST**. Evalúa únicamente sobre el split de validación congelado por M3.

In [1]:
from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec
from argos.config.modeling import load_modeling_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()

m3_gate = paths.docs / "decisions" / "M3_GATE.md"
assert m3_gate.exists() and "M3 ✅ CLOSED" in m3_gate.read_text(encoding="utf-8"), (
    "M3 no está formalmente cerrado. No iniciar Modeling."
)
GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")
MODEL_SPEC = load_modeling_spec(paths.configs / "modeling.toml")
MODEL_DIR = paths.root / "artifacts" / "models" / "m4"
MANIFEST_DIR = paths.root / "artifacts" / "manifests"

{
    "gold_schema": GOLD_SPEC.schema_version,
    "modeling_schema": MODEL_SPEC.schema_version,
    "target": MODEL_SPEC.target,
    "model_matrix": MODEL_SPEC.model_matrix,
    "operational_review_fraction": MODEL_SPEC.ranking.operational_fraction,
}

from argos.models.baselines import BUILD_IMPLEMENTATION
BUILD_IMPLEMENTATION


'm4-30-baselines-v1'

In [2]:
from argos.models.baselines import build_baselines

result = build_baselines(
    root=paths.root,
    gold_spec=GOLD_SPEC,
    modeling_spec=MODEL_SPEC,
    reports_root=paths.reports,
    manifest_dir=MANIFEST_DIR,
)
result.metrics


,model,split,rows,positives,prevalence,pr_auc,roc_auc,log_loss,brier,ece
0,prevalence,validation,950670,9180,0.009656,0.009656,0.500000,0.058330,0.009692,0.011355
1,historical_heuristic,validation,950670,9180,0.009656,0.039690,0.688805,0.138999,0.022798,0.046586


In [3]:
result.ranking

,model,split,review_fraction,reviewed,captured_positives,precision_at_k,recall_at_k,lift_at_k,cumulative_gain
0,prevalence,validation,0.001,951,8,0.008412,0.000871,0.871157,0.000871
1,prevalence,validation,0.005,4754,36,0.007573,0.003922,0.784206,0.003922
2,prevalence,validation,0.010,9507,57,0.005996,0.006209,0.620895,0.006209
3,prevalence,validation,0.025,23767,276,0.011613,0.030065,1.202602,0.030065
4,prevalence,validation,0.050,47534,542,0.011402,0.059041,1.180815,0.059041
5,prevalence,validation,0.100,95067,999,0.010508,0.108824,1.088235,0.108824
6,historical_heuristic,validation,0.001,951,122,0.128286,0.013290,13.285149,0.013290
7,historical_heuristic,validation,0.005,4754,408,0.085822,0.044444,8.887674,0.044444
8,historical_heuristic,validation,0.010,9507,862,0.090670,0.093900,9.389682,0.093900
9,historical_heuristic,validation,0.025,23767,1907,0.080237,0.207734,8.309281,0.207734


In [4]:
result.validation

,check_id,category,blocking,passed,observed,requirement
0,baseline_validation_population,evaluation,True,True,"{""expected"": 950670, ""rows"": 950670}",baseline evaluation covers the complete frozen...
1,baseline_test_isolation,test_isolation,True,True,"{""consumed_splits"": [""train_summary"", ""validat...",notebook 30 does not consume TEST rows
2,baseline_scores_finite,quality,True,True,"{""heuristic_max"": 0.933125, ""heuristic_min"": 0...",baseline scores are finite probabilities
3,baseline_metrics_complete,evaluation,True,True,"{""models"": [""historical_heuristic"", ""prevalenc...",prevalence and historical heuristic baselines ...


In [5]:
blocking = result.validation.loc[result.validation["blocking"]]
print("30 ✅ PASS" if result.ready_for_next_notebook else "30 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Manifest:", result.manifest_path)
assert result.ready_for_next_notebook


30 ✅ PASS
Blocking checks: 4/4
Manifest: /Users/macm1pro/Desktop/projects/argos/argos-code/artifacts/manifests/m4_baselines.json
